# Bayesian Regression Analysis
**Author:** Dipon Konar

This notebook covers two things — a linear regression on a testosterone experiment, and a Poisson regression on syntactic crossings in English vs German. Both done using brms in R.

In [ ]:
if (!requireNamespace("brms", quietly = TRUE)) install.packages("brms")

suppressPackageStartupMessages({
  library(brms)
  library(ggplot2)
  library(dplyr)
})

theme_set(theme_minimal(base_size = 12))
set.seed(42)

---
## Part 1 — Power Posing and Testosterone

The dataset is from a study where participants were assigned to a high or low power pose and testosterone was measured before and after. I'm looking at whether the high-power group showed more increase.

The outcome I care about is `test_diff = testm2 - testm1` (post minus pre testosterone).

In [ ]:
df <- read.table("df_powerpose.csv", header = TRUE, sep = ",")

df$test_diff <- df$testm2 - df$testm1
df$hptreat   <- factor(df$hptreat, levels = c("High", "Low"))

# quick look at group means
df %>%
  group_by(hptreat) %>%
  summarise(n = n(), mean = round(mean(test_diff), 2), sd = round(sd(test_diff), 2))

In [ ]:
ggplot(df, aes(x = hptreat, y = test_diff, fill = hptreat)) +
  geom_boxplot(alpha = 0.5, outlier.shape = NA) +
  geom_jitter(width = 0.1, alpha = 0.5) +
  geom_hline(yintercept = 0, linetype = "dashed", colour = "grey40") +
  scale_fill_manual(values = c("High" = "steelblue", "Low" = "tomato")) +
  labs(title = "Testosterone change by pose group",
       x = "Group", y = "test_diff (pg/ml)") +
  theme(legend.position = "none")

### Model

Simple Gaussian linear model:

$$\text{test\_diff}_i \sim \mathcal{N}(\mu_i, \sigma^2)$$
$$\mu_i = \alpha + \beta_{\text{Low}} \cdot \mathbb{1}[\text{Low}]$$

Used weakly informative priors since I don't have strong prior knowledge about pg/ml ranges:
- Intercept and slope: N(0, 50)
- Sigma: half-Cauchy(0, 10)

In [ ]:
fit1 <- brm(
  test_diff ~ hptreat,
  data   = df,
  family = gaussian(),
  prior  = c(
    prior(normal(0, 50),  class = "Intercept"),
    prior(normal(0, 50),  class = "b"),
    prior(cauchy(0, 10),  class = "sigma")
  ),
  chains = 4, iter = 2000, warmup = 1000, cores = 4,
  seed = 42, refresh = 0, silent = 2
)

summary(fit1)

In [ ]:
# checking if chains converged ok
plot(fit1, variable = c("b_Intercept", "b_hptreatLow", "sigma"))

In [ ]:
pp_check(fit1, ndraws = 80) +
  labs(title = "Posterior predictive check", x = "test_diff")

In [ ]:
plot(conditional_effects(fit1), points = TRUE)

### What I found

The model estimates the Low group had about **8.66 pg/ml less** testosterone increase than the High group (posterior mean for `hptreatLow` ≈ −8.66). The direction supports the hypothesis but the 95% credible interval is [−21.26, 4.39] which crosses zero — so not really conclusive.

The residual SD is around 20 pg/ml which is pretty large compared to the effect, so with n=39 it's hard to say much confidently. Rhat was 1.00 across all parameters so convergence was fine.

---
## Part 2 — Poisson Regression on Crossing Dependencies

Languages tend to avoid crossing dependency arcs. Here I'm modelling the count of crossings per sentence as a Poisson variable that depends on sentence length — and checking whether this differs between English and German.

**M1** — length only (same for both languages)  
**M2** — length × language interaction

The model:
$$N_i \sim \text{Poisson}(\lambda_i), \quad \log\lambda_i = \alpha + \beta L_i$$

### Prior predictive check

Before fitting, I sampled from the priors to see if the predicted crossing counts are even plausible for sentences of length 4.

In [ ]:
simulate_crossings <- function(len, alpha, beta) {
  rpois(1, exp(alpha + beta * len))
}

set.seed(123)
n <- 1000
a <- pmax(0, rnorm(n, 0.15, 0.1))
b <- pmax(0, rnorm(n, 0.25, 0.05))

prior_preds <- mapply(simulate_crossings, len = 4, alpha = a, beta = b)

ggplot(data.frame(x = prior_preds), aes(x = x)) +
  geom_histogram(binwidth = 1, fill = "steelblue", colour = "white") +
  labs(title = "Prior predictions for sentence length = 4",
       x = "Predicted crossings", y = "Count")

Mostly 0–8 which seems fine for short sentences.

### EDA

In [ ]:
obs <- read.table("crossings.csv", sep = ",", header = TRUE)

obs %>%
  group_by(Language, s.length) %>%
  summarise(mean_cross = mean(nCross), .groups = "drop") %>%
  ggplot(aes(x = s.length, y = mean_cross, colour = Language, group = Language)) +
  geom_line() + geom_point() +
  scale_colour_manual(values = c("English" = "steelblue", "German" = "tomato")) +
  labs(title = "Mean crossings by sentence length",
       x = "Sentence length", y = "Mean crossings")

German looks like it grows faster — suggests the interaction in M2 might matter.

In [ ]:
# centre length and code language
obs$s.length_c <- obs$s.length - mean(obs$s.length)
obs$lang       <- as.integer(obs$Language == "German")

### Fitting M1 and M2

In [ ]:
pp <- c(
  prior(normal(0.15, 0.1), class = "Intercept"),
  prior(normal(0, 0.15),   class = "b")
)

fit.m1 <- brm(nCross ~ 1 + s.length_c, data = obs,
              family = poisson(link = "log"), prior = pp,
              chains = 4, iter = 2000, cores = 4, seed = 42, refresh = 0, silent = 2)

fit.m2 <- brm(nCross ~ 1 + s.length_c + lang + s.length_c:lang, data = obs,
              family = poisson(link = "log"), prior = pp,
              chains = 4, iter = 2000, cores = 4, seed = 42, refresh = 0, silent = 2)

cat("M1:\n"); summary(fit.m1)
cat("M2:\n"); summary(fit.m2)

In [ ]:
pp_check(fit.m2, ndraws = 80) + labs(title = "PPC — M2")

In [ ]:
plot(conditional_effects(fit.m2, effects = "s.length_c:lang"), points = TRUE)

### K-fold Cross Validation (k=5)

Using ELPD to see which model predicts better on held-out data.

In [ ]:
K        <- 5
lpds_m1  <- numeric(K)
lpds_m2  <- numeric(K)
untested <- obs

m1_init <- brm(nCross ~ 1 + s.length_c, data = obs,
               family = poisson(link = "log"), prior = pp,
               chains = 1, iter = 1000, warmup = 500, refresh = 0, silent = 2)

m2_init <- brm(nCross ~ 1 + s.length_c + lang + s.length_c:lang, data = obs,
               family = poisson(link = "log"), prior = pp,
               chains = 1, iter = 1000, warmup = 500, refresh = 0, silent = 2)

for (k in 1:K) {
  ytest    <- dplyr::sample_n(untested, size = floor(nrow(obs) / K))
  ytrain   <- dplyr::setdiff(obs, ytest)
  untested <- dplyr::setdiff(untested, ytest)

  m1k <- update(m1_init, newdata = ytrain, refresh = 0, silent = 2)
  m2k <- update(m2_init, newdata = ytrain, refresh = 0, silent = 2)

  p1 <- as.matrix(m1k)
  p2 <- as.matrix(m2k)

  s1 <- 0; s2 <- 0
  for (i in 1:nrow(ytest)) {
    ll1 <- p1[,"b_Intercept"] + p1[,"b_s.length_c"] * ytest$s.length_c[i]
    s1  <- s1 + log(mean(dpois(ytest$nCross[i], exp(ll1))))

    ll2 <- p2[,"b_Intercept"] + p2[,"b_s.length_c"] * ytest$s.length_c[i] +
           p2[,"b_lang"] * ytest$lang[i] +
           p2[,"b_s.length_c:lang"] * ytest$s.length_c[i] * ytest$lang[i]
    s2  <- s2 + log(mean(dpois(ytest$nCross[i], exp(ll2))))
  }

  lpds_m1[k] <- s1
  lpds_m2[k] <- s2
  cat(sprintf("Fold %d — M1: %.1f | M2: %.1f\n", k, s1, s2))
}

cat(sprintf("\nFinal ELPD — M1: %.2f | M2: %.2f | diff: %.2f\n",
            sum(lpds_m1), sum(lpds_m2), sum(lpds_m2) - sum(lpds_m1)))

In [ ]:
data.frame(fold = rep(1:K, 2),
           model = rep(c("M1", "M2"), each = K),
           elpd  = c(lpds_m1, lpds_m2)) %>%
  ggplot(aes(x = factor(fold), y = elpd, fill = model)) +
  geom_col(position = "dodge", width = 0.6) +
  scale_fill_manual(values = c("M1" = "steelblue", "M2" = "seagreen")) +
  labs(title = "ELPD per fold — M1 vs M2", x = "Fold", y = "ELPD")

### Summary

M2 wins by a big margin (~133 ELPD units). This means language actually matters — German accumulates crossings faster than English as sentences get longer, which makes sense given German's freer word order.

---
## Overall Takeaways

**Part 1:** There's a trend in the direction of the power posing hypothesis (~8.7 pg/ml difference) but the CI crosses zero so I can't say it's a clear effect. Sample size is small and variance is high.

**Part 2:** Crossings definitely increase with sentence length. M2 (with the language interaction) predicts held-out data much better than M1, so the growth rate really is different between English and German — not just an artifact.